## Global Runtime Settings

### Objective

Define reusable runtime settings that will be shared across multiple notebook stages.

This setup includes the Melbourne timezone used for audit logs, pipeline timestamps, validation records, and other time-dependent operations.

Using the IANA timezone name `Australia/Melbourne` allows Python to automatically handle the difference between Australian Eastern Standard Time (`AEST`) and Australian Eastern Daylight Time (`AEDT`).


In [ ]:
# ==========================================
# Global Runtime Settings
# ==========================================

from zoneinfo import ZoneInfo


MELBOURNE_TIMEZONE = ZoneInfo("Australia/Melbourne")

## Part 1.1: Load and Verify Project Configuration

### Objective

Load `credentials.json` and `config.json` into the Google Colab runtime and verify that all required configuration variables are present, valid, and ready for use in later pipeline stages.

Because this notebook runs in Google Colab, it cannot directly access files stored in the local project repository. Both JSON files must therefore be uploaded from the local machine into the temporary Colab runtime.

### Configuration Files

* `credentials.json` contains sensitive database connection information, including the database host, database name, user account, and password.
* `config.json` contains reusable project settings, including the Google Drive project path, raw-data location, log directory, schema-registry filenames, type-rule filename, audit-log filename, and profiler sample size.

### Validation Rules

This step verifies that:

* both required JSON files are uploaded;
* both files contain valid JSON objects;
* all required credential and configuration keys are present;
* required values are not empty;
* `BASE_DIR` is a complete Colab Google Drive path;
* `RAW_DATA_PATH` and `LOG_AUDIT_PATH` are relative directory names;
* schema, rule, and audit-log settings contain filenames rather than directory paths;
* `PROFILE_SAMPLE_ROWS` is either `null`, `0`, or a positive integer.

This validation prevents later notebook sections from failing because of missing variables, invalid paths, or inconsistent configuration values.

### Process

This step will:

1. Prompt the user to upload `credentials.json` and `config.json`.
2. Confirm that both required files were uploaded.
3. Load the JSON contents into the `credentials` and `config` dictionaries.
4. Validate all required keys and values.
5. Validate project-specific path and filename rules.
6. Display the loaded configuration for verification.
7. Mask sensitive credential values before displaying them.
8. Remove the uploaded JSON files from the temporary Colab filesystem after loading them into memory.

Removing the temporary files reduces unnecessary credential exposure within the Colab runtime. The loaded `credentials` and `config` dictionaries remain available in memory until the Colab runtime is restarted or disconnected.

### Expected Outcome

* Both JSON files are uploaded and loaded successfully.
* All required credential and configuration keys are present.
* All configuration values pass validation.
* Sensitive credential values are not displayed in plain text.
* Temporary JSON files are removed from the Colab runtime.
* The `credentials` and `config` dictionaries remain available for Parts 1.2, 1.3, and later pipeline stages.


In [ ]:
# ==========================================
# Part 1.1: Upload, Load, and Verify
# Project Configuration in Google Colab
# ==========================================

import json
from pathlib import Path

from google.colab import files


# ------------------------------------------
# Required files and configuration keys
# ------------------------------------------

REQUIRED_FILES = {
    "credentials.json",
    "config.json",
}

REQUIRED_CREDENTIAL_KEYS = {
    "DB_HOST",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD",
}

REQUIRED_CONFIG_KEYS = {
    "BASE_DIR",
    "RAW_DATA_PATH",
    "LOG_AUDIT_PATH",
    "MASTER_SCHEMA",
    "PY_SCHEMA",
    "TYPE_RULES",
    "PIPELINE_AUDIT",
    "PROFILE_SAMPLE_ROWS",
}

CONFIG_FILENAME_KEYS = {
    "MASTER_SCHEMA",
    "PY_SCHEMA",
    "TYPE_RULES",
    "PIPELINE_AUDIT",
}


def upload_configuration_files() -> None:
    """
    Prompt the user to upload credentials.json and config.json
    from the local machine into the temporary Colab runtime.
    """
    print("Upload the following files:")
    print("- credentials.json")
    print("- config.json")
    print()

    uploaded_files = files.upload()
    uploaded_names = set(uploaded_files.keys())

    missing_files = REQUIRED_FILES - uploaded_names

    if missing_files:
        raise FileNotFoundError(
            "Missing required file(s): "
            + ", ".join(sorted(missing_files))
        )

    print("\nConfiguration files uploaded successfully.")


def load_json(file_path: Path) -> dict:
    """
    Load a JSON file and return its contents as a dictionary.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        with file_path.open("r", encoding="utf-8") as file:
            data = json.load(file)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON format in {file_path.name}: {error}"
        ) from error

    if not isinstance(data, dict):
        raise TypeError(
            f"{file_path.name} must contain a JSON object."
        )

    return data


def validate_required_keys(
    data: dict,
    required_keys: set[str],
    file_name: str,
) -> None:
    """
    Confirm that all required keys exist in the loaded JSON object.
    """
    missing_keys = required_keys - data.keys()

    if missing_keys:
        raise KeyError(
            f"Missing required key(s) in {file_name}: "
            + ", ".join(sorted(missing_keys))
        )


def validate_non_empty_values(
    data: dict,
    required_keys: set[str],
    file_name: str,
) -> None:
    """
    Confirm that required values are not null or empty strings.
    """
    empty_keys = [
        key
        for key in required_keys
        if data.get(key) is None
        or (
            isinstance(data.get(key), str)
            and not data.get(key).strip()
        )
    ]

    if empty_keys:
        raise ValueError(
            f"Empty required value(s) in {file_name}: "
            + ", ".join(sorted(empty_keys))
        )


def validate_project_config(config_data: dict) -> None:
    """
    Validate project-specific configuration rules.
    """
    base_dir = config_data["BASE_DIR"]

    expected_drive_prefix = "/content/drive/MyDrive/"

    if not isinstance(base_dir, str):
        raise TypeError("BASE_DIR must be a string.")

    if not base_dir.startswith(expected_drive_prefix):
        raise ValueError(
            "BASE_DIR must be a complete Colab Google Drive path "
            f"starting with '{expected_drive_prefix}'."
        )

    relative_directory_keys = {
        "RAW_DATA_PATH",
        "LOG_AUDIT_PATH",
    }

    for key in relative_directory_keys:
        value = config_data[key]

        if not isinstance(value, str):
            raise TypeError(f"{key} must be a string.")

        configured_path = Path(value)

        if configured_path.is_absolute():
            raise ValueError(
                f"{key} must be relative to BASE_DIR, not an absolute path."
            )

    for key in CONFIG_FILENAME_KEYS:
        value = config_data[key]

        if not isinstance(value, str):
            raise TypeError(f"{key} must be a string.")

        configured_file = Path(value)

        if configured_file.name != value:
            raise ValueError(
                f"{key} must contain a filename only, not a directory path."
            )

    sample_rows = config_data["PROFILE_SAMPLE_ROWS"]

    if sample_rows is not None and not isinstance(sample_rows, int):
        raise TypeError(
            "PROFILE_SAMPLE_ROWS must be an integer or null."
        )

    if isinstance(sample_rows, int) and sample_rows < 0:
        raise ValueError(
            "PROFILE_SAMPLE_ROWS cannot be negative."
        )


def mask_sensitive_value(key: str, value):
    """
    Mask sensitive values before displaying them.
    """
    sensitive_terms = (
        "PASS",
        "PASSWORD",
        "SECRET",
        "TOKEN",
        "API_KEY",
        "PRIVATE_KEY",
    )

    key_upper = key.upper()

    if any(term in key_upper for term in sensitive_terms):
        return "********"

    return value


def display_configuration(
    file_name: str,
    data: dict,
    mask_sensitive: bool = False,
) -> None:
    """
    Print configuration variables for verification.
    """
    print(f"========== {file_name} ==========")

    for key, value in data.items():
        displayed_value = (
            mask_sensitive_value(key, value)
            if mask_sensitive
            else value
        )

        print(f"{key:<25}: {displayed_value}")

    print()


def remove_runtime_file(file_path: Path) -> None:
    """
    Delete an uploaded file from the temporary Colab filesystem.
    """
    if file_path.exists():
        file_path.unlink()
        print(f"Removed temporary file: {file_path.name}")


# ------------------------------------------
# Upload files from local machine
# ------------------------------------------

upload_configuration_files()


# ------------------------------------------
# Define temporary Colab file paths
# ------------------------------------------

CREDENTIALS_FILE = Path("/content/credentials.json")
CONFIG_FILE = Path("/content/config.json")


# ------------------------------------------
# Load JSON configuration into memory
# ------------------------------------------

credentials = load_json(CREDENTIALS_FILE)
config = load_json(CONFIG_FILE)

print("\ncredentials.json loaded successfully.")
print("config.json loaded successfully.\n")


# ------------------------------------------
# Validate required keys and values
# ------------------------------------------

validate_required_keys(
    data=credentials,
    required_keys=REQUIRED_CREDENTIAL_KEYS,
    file_name="credentials.json",
)

validate_non_empty_values(
    data=credentials,
    required_keys=REQUIRED_CREDENTIAL_KEYS,
    file_name="credentials.json",
)

validate_required_keys(
    data=config,
    required_keys=REQUIRED_CONFIG_KEYS,
    file_name="config.json",
)

# PROFILE_SAMPLE_ROWS may intentionally be null or 0,
# so it is excluded from the generic non-empty check.
required_non_empty_config_keys = (
    REQUIRED_CONFIG_KEYS - {"PROFILE_SAMPLE_ROWS"}
)

validate_non_empty_values(
    data=config,
    required_keys=required_non_empty_config_keys,
    file_name="config.json",
)

validate_project_config(config)

print("✓ Required database credentials verified.")
print("✓ Required project configuration verified.")
print("✓ Project configuration values passed validation.\n")


# ------------------------------------------
# Display loaded variables
# ------------------------------------------

display_configuration(
    file_name="credentials.json",
    data=credentials,
    mask_sensitive=True,
)

display_configuration(
    file_name="config.json",
    data=config,
    mask_sensitive=False,
)


# ------------------------------------------
# Remove uploaded files from Colab storage
# ------------------------------------------

remove_runtime_file(CREDENTIALS_FILE)
remove_runtime_file(CONFIG_FILE)


# ------------------------------------------
# Final verification
# ------------------------------------------

print()
print("=" * 65)
print("✓ Project configuration loaded into memory successfully.")
print("✓ Required variables and values were verified.")
print("✓ Temporary JSON files were removed from the Colab runtime.")
print("✓ Configuration remains accessible for subsequent steps.")
print("=" * 65)

## Part 1.2: Mount Google Drive, Verify Project Paths, and Record the Connection

### Objective

Mount Google Drive in the Google Colab runtime, resolve the project paths defined in `config.json`, verify that the required project directories are available, and record the successful environment setup in the pipeline audit log.

The configuration loaded in Part 1.1 is converted into reusable path objects that will be shared by all subsequent notebook stages.

### Expected Project Structure

```text
BASE_DIR/
├── OriginalDataFile/
└── logs/
    ├── pipeline_audit.log
    ├── master_schema.json
    ├── py_schema.json
    └── type_rules.json
```

### Configuration Variables

The following configuration variables are used to construct the project paths:

* `BASE_DIR` identifies the project root directory in Google Drive.
* `RAW_DATA_PATH` identifies the source data directory relative to `BASE_DIR`.
* `LOG_AUDIT_PATH` identifies the runtime log directory relative to `BASE_DIR`.
* `MASTER_SCHEMA` identifies the SQL schema registry filename.
* `PY_SCHEMA` identifies the Pandas schema registry filename.
* `TYPE_RULES` identifies the profiler type-rule filename.
* `PIPELINE_AUDIT` identifies the pipeline audit-log filename.
* `PROFILE_SAMPLE_ROWS` defines the default profiling sample size for later pipeline stages.

These values are resolved into reusable path variables:

```text
BASE_DIR_PATH
RAW_DATA_PATH
LOG_AUDIT_PATH

MASTER_SCHEMA_PATH
PY_SCHEMA_PATH
TYPE_RULES_PATH
PIPELINE_AUDIT_PATH
```

### Process

This step will:

1. Mount Google Drive at `/content/drive`.
2. Read the required project configuration from the `config` dictionary.
3. Resolve the configured directory and file paths.
4. Verify that `BASE_DIR_PATH` exists and is a directory.
5. Verify that `RAW_DATA_PATH` exists and is a directory.
6. Create `LOG_AUDIT_PATH` if it does not already exist.
7. Resolve the runtime file locations used by the profiler.
8. Create `pipeline_audit.log` if it does not already exist.
9. Append a timestamped success record to the pipeline audit log using the Melbourne timezone.

The `logs` directory is managed by the pipeline and is created automatically when missing. The profiler runtime files (`master_schema.json`, `py_schema.json`, and `type_rules.json`) are **not** created in this step. They will be initialised during Part 2, where their default structures and validation rules are defined.

### Audit Record

A successful verification appends a record similar to:

```text
2026-07-15 19:30:00 AEST | INFO | PART_1.2 | Google Drive mounted and project paths verified successfully.
```

The audit log is opened in append mode so that previous execution records are preserved.

### Expected Outcome

* Google Drive is mounted successfully.
* `BASE_DIR_PATH` points to an existing project directory.
* `RAW_DATA_PATH` points to an existing source-data directory.
* `LOG_AUDIT_PATH` exists and is ready to store runtime artifacts.
* Runtime file paths are resolved for later pipeline stages.
* `pipeline_audit.log` exists and is ready to receive audit records.
* A timestamped success record is appended to the audit log.
* The notebook is ready for database validation and subsequent pipeline stages.


In [ ]:
# ==========================================
# Part 1.2: Mount Google Drive, Verify Paths,
# and Record Successful Connection
# ==========================================

from datetime import datetime
from pathlib import Path

from google.colab import drive


# ------------------------------------------
# Mount Google Drive
# ------------------------------------------

DRIVE_MOUNT_POINT = Path("/content/drive")

drive.mount(str(DRIVE_MOUNT_POINT))

print("\n✓ Google Drive mounted successfully.\n")


# ------------------------------------------
# Verify required configuration variables
# ------------------------------------------

REQUIRED_CONFIG_KEYS = {
    "BASE_DIR",
    "RAW_DATA_PATH",
    "LOG_AUDIT_PATH",
    "MASTER_SCHEMA",
    "PY_SCHEMA",
    "TYPE_RULES",
    "PIPELINE_AUDIT",
    "PROFILE_SAMPLE_ROWS",
}

missing_config_keys = REQUIRED_CONFIG_KEYS - config.keys()

if missing_config_keys:
    raise KeyError(
        "Missing required config variable(s): "
        + ", ".join(sorted(missing_config_keys))
    )


# ------------------------------------------
# Read project configuration
# ------------------------------------------

BASE_DIR = config["BASE_DIR"]
RAW_DATA_DIR = config["RAW_DATA_PATH"]
LOG_AUDIT_DIR = config["LOG_AUDIT_PATH"]

MASTER_SCHEMA = config["MASTER_SCHEMA"]
PY_SCHEMA = config["PY_SCHEMA"]
TYPE_RULES = config["TYPE_RULES"]
PIPELINE_AUDIT = config["PIPELINE_AUDIT"]

PROFILE_SAMPLE_ROWS = config["PROFILE_SAMPLE_ROWS"]


# ------------------------------------------
# Resolve project paths
# ------------------------------------------

# BASE_DIR is already configured as a complete Colab path.
BASE_DIR_PATH = Path(BASE_DIR)

# Project directories
RAW_DATA_PATH = BASE_DIR_PATH / RAW_DATA_DIR
LOG_AUDIT_PATH = BASE_DIR_PATH / LOG_AUDIT_DIR

# Runtime files stored inside the logs directory
MASTER_SCHEMA_PATH = LOG_AUDIT_PATH / MASTER_SCHEMA
PY_SCHEMA_PATH = LOG_AUDIT_PATH / PY_SCHEMA
TYPE_RULES_PATH = LOG_AUDIT_PATH / TYPE_RULES
PIPELINE_AUDIT_PATH = LOG_AUDIT_PATH / PIPELINE_AUDIT


# ------------------------------------------
# Verify required project directories
# ------------------------------------------

required_directories = {
    "BASE_DIR_PATH": BASE_DIR_PATH,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
}

print("========== Google Drive Path Verification ==========\n")

# BASE_DIR must already exist.
if not BASE_DIR_PATH.exists():
    raise FileNotFoundError(
        f"BASE_DIR_PATH does not exist:\n{BASE_DIR_PATH}"
    )

if not BASE_DIR_PATH.is_dir():
    raise NotADirectoryError(
        f"BASE_DIR_PATH is not a directory:\n{BASE_DIR_PATH}"
    )

print("✓ BASE_DIR_PATH verified")
print(f"  {BASE_DIR_PATH}\n")


# RAW_DATA_PATH must already exist because the pipeline
# should not silently create an empty source-data directory.
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"RAW_DATA_PATH does not exist:\n{RAW_DATA_PATH}"
    )

if not RAW_DATA_PATH.is_dir():
    raise NotADirectoryError(
        f"RAW_DATA_PATH is not a directory:\n{RAW_DATA_PATH}"
    )

print("✓ RAW_DATA_PATH verified")
print(f"  {RAW_DATA_PATH}\n")


# LOG_AUDIT_PATH is pipeline-managed, so create it when missing.
if LOG_AUDIT_PATH.exists():

    if not LOG_AUDIT_PATH.is_dir():
        raise NotADirectoryError(
            f"LOG_AUDIT_PATH exists but is not a directory:\n"
            f"{LOG_AUDIT_PATH}"
        )

    print("✓ LOG_AUDIT_PATH found")
    print(f"  {LOG_AUDIT_PATH}\n")

else:
    LOG_AUDIT_PATH.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("✓ LOG_AUDIT_PATH created")
    print(f"  {LOG_AUDIT_PATH}\n")


# ------------------------------------------
# Verify runtime file targets
# ------------------------------------------

runtime_file_paths = {
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
}

print("========== Runtime File Verification ==========\n")

for variable_name, file_path in runtime_file_paths.items():

    if file_path.exists() and not file_path.is_file():
        raise FileExistsError(
            f"{variable_name} exists but is not a file:\n"
            f"{file_path}"
        )

    status = (
        "EXISTS"
        if file_path.exists()
        else "NOT CREATED YET"
    )

    print(f"✓ {variable_name}")
    print(f"  {file_path}")
    print(f"  Status: {status}\n")


# ------------------------------------------
# Verify profiling sample configuration
# ------------------------------------------

if (
    PROFILE_SAMPLE_ROWS is not None
    and not isinstance(PROFILE_SAMPLE_ROWS, int)
):
    raise TypeError(
        "PROFILE_SAMPLE_ROWS must be an integer or null."
    )

if (
    isinstance(PROFILE_SAMPLE_ROWS, int)
    and PROFILE_SAMPLE_ROWS < 0
):
    raise ValueError(
        "PROFILE_SAMPLE_ROWS cannot be negative."
    )

profiling_mode = (
    "FULL_FILE"
    if PROFILE_SAMPLE_ROWS in (None, 0)
    else "SAMPLED"
)

print("========== Profiling Configuration ==========\n")
print(f"✓ Profiling mode : {profiling_mode}")
print(f"✓ Sample rows    : {PROFILE_SAMPLE_ROWS}\n")


# ------------------------------------------
# Verify or create pipeline audit log
# ------------------------------------------

if PIPELINE_AUDIT_PATH.exists():

    if not PIPELINE_AUDIT_PATH.is_file():
        raise FileExistsError(
            "PIPELINE_AUDIT_PATH exists but is not a file:\n"
            f"{PIPELINE_AUDIT_PATH}"
        )

    print("✓ Pipeline audit log found")
    print(f"  {PIPELINE_AUDIT_PATH}\n")

else:
    PIPELINE_AUDIT_PATH.touch()

    print("✓ Pipeline audit log created")
    print(f"  {PIPELINE_AUDIT_PATH}\n")


# ------------------------------------------
# Write successful connection audit record
# ------------------------------------------

timestamp = datetime.now(
    MELBOURNE_TIMEZONE
).strftime("%Y-%m-%d %H:%M:%S %Z")

audit_message = (
    f"[{timestamp}] "
    f"[RUN: - ] "
    f"[INFO] "
    f"PART_1.2 | Google Drive mounted and project paths verified successfully."
)

with PIPELINE_AUDIT_PATH.open(
    mode="a",
    encoding="utf-8",
) as audit_file:
    # Append rather than overwrite so previous pipeline
    # executions remain auditable.
    audit_file.write(audit_message + "\n")


print("✓ Successful connection recorded in pipeline audit log.")
print(f"  {audit_message}\n")


# ------------------------------------------
# Final summary
# ------------------------------------------

print("=" * 70)
print("✓ Google Drive connection completed.")
print("✓ Required project directories verified.")
print("✓ Runtime file paths resolved.")
print("✓ Profiling configuration verified.")
print("✓ Pipeline audit log updated.")
print("=" * 70)

## Part 1.3: Verify Database Connection

### Objective

Verify that the notebook can establish a secure connection to the configured PostgreSQL database before running any queries or data-processing operations.

The database credentials loaded in Part 1.1 are assigned to reusable variables:

* `DB_HOST` identifies the database server.
* `DB_NAME` identifies the target database.
* `DB_USER` identifies the database user account.
* `DB_PASSWORD` contains the database password.
* `DB_PORT` identifies the PostgreSQL connection port.
* `DB_SSLMODE` controls SSL encryption for the database connection.

### Process

This step will:

1. Read the required database credentials from the `credentials` dictionary.
2. Validate that the required credential values are present.
3. Establish a PostgreSQL connection using a connection timeout.
4. Run a small validation query to confirm that the database session is operational.
5. Display the connected database name and user without exposing the password.
6. Close the cursor and database connection after verification.
7. Append a timestamped success record to `pipeline_audit.log` using Melbourne time.

The database connection is used only for verification in this step and is closed immediately after the test completes. Later pipeline stages should create and manage their own database connections as required.

### Audit Record

A successful connection writes a record similar to:

```text
2026-07-15 19:30:00 AEST | INFO | PART_1.3 | PostgreSQL database connection verified successfully.
```

### Expected Outcome

* All required database credentials are accessible.
* The PostgreSQL connection is established successfully.
* The validation query executes successfully.
* The database connection is closed safely.
* A timestamped success record is appended to the pipeline audit log.
* The notebook is ready for subsequent database queries and data-processing steps.


In [ ]:
# ==========================================
# Part 1.3: Verify Database Connection
# ==========================================

from datetime import datetime

import psycopg2
from psycopg2 import OperationalError


# ------------------------------------------
# Read database credentials
# ------------------------------------------

REQUIRED_DATABASE_KEYS = {
    "DB_HOST",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD",
}

missing_database_keys = REQUIRED_DATABASE_KEYS - credentials.keys()

if missing_database_keys:
    raise KeyError(
        "Missing required database credential(s): "
        + ", ".join(sorted(missing_database_keys))
    )


DB_HOST = credentials["DB_HOST"]
DB_NAME = credentials["DB_NAME"]
DB_USER = credentials["DB_USER"]
DB_PASSWORD = credentials["DB_PASSWORD"]

# Optional settings with PostgreSQL defaults
DB_PORT = int(credentials.get("DB_PORT", 5432))
DB_SSLMODE = credentials.get("DB_SSLMODE", "require")


# ------------------------------------------
# Validate credential values
# ------------------------------------------

database_settings = {
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
}

empty_database_values = [
    key
    for key, value in database_settings.items()
    if value is None or str(value).strip() == ""
]

if empty_database_values:
    raise ValueError(
        "Empty database credential value(s): "
        + ", ".join(empty_database_values)
    )


# ------------------------------------------
# Verify database connection
# ------------------------------------------

connection = None
cursor = None

try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT
            current_database(),
            current_user,
            version();
        """
    )

    connected_database, connected_user, database_version = cursor.fetchone()

    print("========== Database Connection Verification ==========\n")
    print("✓ PostgreSQL connection established successfully.")
    print(f"  Database : {connected_database}")
    print(f"  User     : {connected_user}")
    print(f"  Host     : {DB_HOST}")
    print(f"  Port     : {DB_PORT}")
    print(f"  SSL mode : {DB_SSLMODE}")
    print(f"  Version  : {database_version.split(',')[0]}\n")


    # ------------------------------------------
    # Write successful connection audit record
    # ------------------------------------------

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).strftime("%Y-%m-%d %H:%M:%S %Z")

    audit_message = (
        f"[{timestamp}] "
        f"[RUN: - ] "
        f"[INFO] "
        f"PART_1.3 | PostgreSQL database connection verified successfully."
    )

    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(audit_message + "\n")

    print("✓ Successful database connection recorded in pipeline audit log.")
    print(f"  {audit_message}\n")


except OperationalError as error:
    raise ConnectionError(
        "Unable to connect to the PostgreSQL database. "
        "Check DB_HOST, DB_NAME, DB_USER, DB_PASSWORD, network access, "
        "firewall rules, and SSL configuration."
    ) from error


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


print("\n" + "=" * 65)
print("✓ Database connection verification completed.")
print("=" * 65)